# Read actual asset verification report
Read-only acceptance of downloaded assets and synthetic checkpoint checks; preflight runs without model inference.

In [1]:
from pathlib import Path
import json, subprocess
root=Path('/content/MRI-classification')
drive=Path('/content/drive/MyDrive/FYP_MRI')
reports=sorted((drive/'artifacts/reports').glob('encoder_assets_*.json'))
assert reports, 'Asset preparation did not produce a final report'
p=reports[-1]; report=json.loads(p.read_text())
assert report['patient_data_used'] is False and report['training'] is False
assert report['brainiac_check']['strict_backbone_load'] is True
assert all(x['strict_load'] and x['finite'] for x in report['brain_jepa_check']['branches'])
for asset in report['assets'].values():
    q=Path(asset['path']); assert q.is_file() and q.stat().st_size==asset['size_bytes']
print('FINAL ASSET REPORT',p)
print(json.dumps({k:v for k,v in report.items() if k!='brain_jepa_environment'},ensure_ascii=False,indent=2))
for encoder,python in [('brainiac','/content/venvs/brainiac/bin/python'),('brain_jepa','/content/venvs/brain_jepa/bin/python')]:
    result=subprocess.run([python,str(root/'scripts/preflight.py'),'--encoder',encoder,'--paths',str(root/'configs/paths.local.yaml'),'--config',str(root/'configs/pilot.local.yaml')],capture_output=True,text=True,cwd=root)
    assert result.returncode in (0,2),result.stderr
    result_json=json.loads(result.stdout)
    assert not any('snapshot verification failed' in x.lower() for x in result_json['missing'])
    print('PREFLIGHT',encoder,json.dumps({'status':result_json['status'],'missing':result_json['missing']},ensure_ascii=False))
print('Real QC and agreed recipes required; no participant features or classification results exist.')


FINAL ASSET REPORT /content/drive/MyDrive/FYP_MRI/artifacts/reports/encoder_assets_20261108_110935_1794136175552854375.json
{
  "utc_stamp": "20261108_110935_1794136175552854375",
  "patient_data_used": false,
  "training": false,
  "assets": {
    "brainiac": {
      "path": "/content/drive/MyDrive/FYP_MRI/weights/brainiac/BrainIAC.ckpt",
      "size_bytes": 362119240,
      "sha256": "f22bdbcae26823a9d9e8aee883c6f24386ba4617339c12269848b6666cc62693",
      "source": "https://www.dropbox.com/scl/fo/i51xt63roognvt7vuslbl/AMblt6reQVvlSrORTB3_2lE/BrainIAC.ckpt?rlkey=9w55le6tslwxlfz6c0viylmjb&dl=1"
    },
    "brainiac_template": {
      "path": "/content/drive/MyDrive/FYP_MRI/weights/reference/brainiac_temp_head.nii.gz",
      "size_bytes": 11976113,
      "sha256": "d8eba10e9528a5212b78fb1be477d5e8229751bfec5e1e1a648dc81ff83a622e",
      "source": "https://raw.githubusercontent.com/AIM-KannLab/BrainIAC/ba60f45bed832ee0e5683c678caeaeefe2072f0d/src/preprocessing/atlases/temp_head.nii.gz"
